# Campus Access — YOLO + ByteTrack on Colab T4

Runs the entry/exit counter pipeline on Colab's free T4 GPU. Uses NVIDIA API key for LLM analysis of results.

In [ ]:
# Install dependencies
!pip install -q ultralytics supervision opencv-python onnxruntime 2>&1 | tail -5

In [ ]:
# Check GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Download sample videos from GitHub
import urllib.request
import os

repo_base = "https://github.com/Rawbeew/campus-access/raw/main/"
videos = ["sample_input_pedestrian.mp4", "sample_input_vehicle.mp4"]

for v in videos:
    if not os.path.exists(v):
        print(f'Downloading {v}...')
        urllib.request.urlretrieve(repo_base + v, v)
        print(f'  Done: {os.path.getsize(v)/1e6:.1f} MB')
    else:
        print(f'Already have {v}')

In [ ]:
# Run pipeline on pedestrian video
import cv2
import numpy as np
from ultralytics import YOLO
import supervision as sv
import json
import time

# Load model (will use GPU automatically)
model = YOLO('yolov8n.pt')
model.to('cuda')

# Initialize trackers
ped_tracker = sv.ByteTrack()
veh_tracker = sv.ByteTrack()

# Line zones (same as local config)
ped_line = sv.LineZone(start=sv.Point(100, 100), end=sv.Point(500, 100))
veh_line = sv.LineZone(start=sv.Point(100, 200), end=sv.Point(500, 200))

# Annotators
box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_thickness=1, text_scale=0.5)
line_annotator = sv.LineZoneAnnotator(thickness=2, text_thickness=1, text_scale=0.5)

cap = cv2.VideoCapture('sample_input_pedestrian.mp4')
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output_pedestrian_colab.mp4', fourcc, fps, (width, height))

ped_in = 0
ped_out = 0
frame_idx = 0
start = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # YOLO inference
    results = model(frame, verbose=False, conf=0.4)[0]
    detections = sv.Detections.from_ultralytics(results)
    
    # Filter for person class (0)
    detections = detections[detections.class_id == 0]
    
    # Track
    tracked = ped_tracker.update_with_detections(detections)
    
    # Line zone
    ped_line.trigger(tracked)
    
    # Annotate
    labels = [f"#{tid}" for tid in tracked.tracker_id] if tracked.tracker_id is not None else []
    annotated = box_annotator.annotate(scene=frame.copy(), detections=tracked)
    annotated = label_annotator.annotate(scene=annotated, detections=tracked, labels=labels)
    annotated = line_annotator.annotate(annotated, ped_line)
    
    out.write(annotated)
    frame_idx += 1
    
    if frame_idx % 50 == 0:
        print(f'  Frame {frame_idx}/{total_frames}')

cap.release()
out.release()

ped_in = ped_line.in_count
ped_out = ped_line.out_count
elapsed = time.time() - start

print(f'Pedestrian video done in {elapsed:.1f}s ({total_frames/elapsed:.1f} FPS)')
print(f'  IN: {ped_in}, OUT: {ped_out}')

ped_results = {'in': ped_in, 'out': ped_out, 'frames': frame_idx, 'time_s': elapsed}

In [ ]:
# Run pipeline on vehicle video
veh_tracker = sv.ByteTrack()

cap = cv2.VideoCapture('sample_input_vehicle.mp4')
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FPS))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output_vehicle_colab.mp4', fourcc, fps, (width, height))

veh_in = 0
veh_out = 0
frame_idx = 0
start = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    results = model(frame, verbose=False, conf=0.4)[0]
    detections = sv.Detections.from_ultralytics(results)
    
    # Filter for vehicle classes (2=car, 3=motorcycle, 5=bus, 7=truck)
    vehicle_classes = [2, 3, 5, 7]
    mask = np.isin(detections.class_id, vehicle_classes)
    detections = detections[mask]
    
    tracked = veh_tracker.update_with_detections(detections)
    
    veh_line.trigger(tracked)
    
    labels = [f"#{tid}" for tid in tracked.tracker_id] if tracked.tracker_id is not None else []
    annotated = box_annotator.annotate(scene=frame.copy(), detections=tracked)
    annotated = label_annotator.annotate(scene=annotated, detections=tracked, labels=labels)
    annotated = line_annotator.annotate(annotated, veh_line)
    
    out.write(annotated)
    frame_idx += 1
    
    if frame_idx % 50 == 0:
        print(f'  Frame {frame_idx}/{total_frames}')

cap.release()
out.release()

veh_in = veh_line.in_count
veh_out = veh_line.out_count
elapsed = time.time() - start

print(f'Vehicle video done in {elapsed:.1f}s ({total_frames/elapsed:.1f} FPS)')
print(f'  IN: {veh_in}, OUT: {veh_out}')

veh_results = {'in': veh_in, 'out': veh_out, 'frames': frame_idx, 'time_s': elapsed}

In [ ]:
# Save results JSON
results = {
    "pedestrian": ped_results,
    "vehicle": veh_results,
    "summary": {
        "total_pedestrians": ped_results['in'] + ped_results['out'],
        "total_vehicles": veh_results['in'] + veh_results['out'],
        "ped_in": ped_results['in'],
        "ped_out": ped_results['out'],
        "veh_in": veh_results['in'],
        "veh_out": veh_results['out']
    }
}

with open('campus_access_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))

In [ ]:
# LLM Analysis using NVIDIA API key
import os
import requests
import json

# Use NVIDIA API key from env
NVIDIA_API_KEY = os.environ.get('NVIDIA_API_KEY', '')

if not NVIDIA_API_KEY:
    print("NVIDIA_API_KEY not set in Colab env. Set it in Colab secrets or pass manually.")
else:
    prompt = f"""Analyze these campus entry/exit counting results:

Pedestrian gate: {ped_results['in']} entries, {ped_results['out']} exits
Vehicle gate: {veh_results['in']} entries, {veh_results['out']} exits

Provide:
1. Net flow (entries - exits) for each
2. Peak direction assessment
3. Any anomaly flags (e.g., more exits than entries = possible count error)
4. One-sentence operational summary for facilities team
"""
    
    headers = {
        "Authorization": f"Bearer {NVIDIA_API_KEY}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": "nvidia/nemotron-3-ultra",
        "messages": [
            {"role": "system", "content": "You are a facilities analytics assistant. Be concise, factual, no fluff."},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.1,
        "max_tokens": 300
    }
    
    response = requests.post(
        "https://integrate.api.nvidia.com/v1/chat/completions",
        headers=headers,
        json=payload,
        timeout=30
    )
    
    if response.status_code == 200:
        analysis = response.json()['choices'][0]['message']['content']
        print("=== LLM ANALYSIS ===")
        print(analysis)
        
        with open('llm_analysis.txt', 'w') as f:
            f.write(analysis)
    else:
        print(f"API error: {response.status_code}")
        print(response.text)

In [ ]:
# Download outputs
from google.colab import files
files.download('output_pedestrian_colab.mp4')
files.download('output_vehicle_colab.mp4')
files.download('campus_access_results.json')
files.download('llm_analysis.txt')